# Dual Arm lifting a box using Cartesian Impedance Control

In [1]:
import os
file_path = os.path.abspath(".")
print(file_path)


import sys
print(sys.executable)

import pinocchio as pin
import mujoco
import mujoco.viewer

import numpy as np

# get current file
urdf = file_path + "/model/bifrank_robot.urdf"

urdf = os.path.join(file_path, "model", "bifrank_robot.urdf")

model_full = pin.buildModelFromUrdf(urdf)
data_full = model_full.createData()
q0 = pin.neutral(model_full)

model = pin.buildReducedModel(model_full, [], q0)
data = model.createData()

## Get end-effector numbers
# The frame id of the end-effector
end_effector_link_name = ["lewis_fr3_link7", "richard_fr3_link7"]

#TODO: using for Loop to check and get the frame id of end-effectors and store them in a list
#[ ]: check the end-effector names in the urdf file
#[ ]: get the frame ids and store them in a list 
left_ee_frame_id = model.getFrameId(end_effector_link_name[0])
right_ee_frame_id = model.getFrameId(end_effector_link_name[1])
print("left_ee_frame_id: ", left_ee_frame_id, "right_ee_frame_id: ", right_ee_frame_id)
print("DoF:", model.nq, model.nv)


mjcf_path = os.path.join(file_path, "model", "scene.xml")

model_mj = mujoco.MjModel.from_xml_path(mjcf_path)
data_mj = mujoco.MjData(model_mj)

mujoco.mj_forward(model_mj, data_mj)
print("MuJoCo nq =", model_mj.nq, " nv =", model_mj.nv, " nu =", model_mj.nu)

viewer = mujoco.viewer.launch_passive(model_mj, data_mj)


/home/dong/Documents/PhD_Dualarm/dualarm-python-test
/home/dong/miniconda3/envs/dualarm-control/bin/python
left_ee_frame_id:  21 right_ee_frame_id:  41
DoF: 15 15
MuJoCo nq = 22  nv = 21  nu = 15


In [2]:
# pinocchio model states
joint_indices = np.arange(0, model.nq)

joint_initial_pos = [0.0, 0.0, -0.7854, 0.0, -2.35621, -0.7854, 1.5708, 0.0, 
                          0.0, -0.7854, 0.0, -2.35621, 0.7854, 1.5708,  -1.5708]
# joint_initial_pos = [0.0, 0.0, -0.7854, 0.0, -2.35621, -0.7854, 1.5708, 0.785398, 
                        #   0.0, -0.7854, 0.0, -2.35621, 0.7854, 1.5708, -1.5708]
data_mj.qpos[joint_indices] = joint_initial_pos
data_mj.qvel[joint_indices] = np.zeros_like(model_mj.nv)
mujoco.mj_forward(model_mj, data_mj)
mujoco.mj_step(model_mj, data_mj)
viewer.sync()

left_des_R = data.oMf[left_ee_frame_id].rotation.copy()
right_des_R = data.oMf[right_ee_frame_id].rotation.copy()

# print("Mujoco data_mj.qvel: ", data_mj.qvel)
# print("MuJoCo qpos after setting initial pos =", data_mj.qpos)

# The number of pin_joints should match the number of mujoco joints
assert model.nq == model_mj.nu, "Mismatch in DoF between Pinocchio and MuJoCo models"
def get_pin_state_from_mujoco():
    q  = data_mj.qpos[joint_indices].copy()
    dq = data_mj.qvel[joint_indices].copy()
    return q, dq

def rotation_error(R_des, R):
    """
    Compute orientation error e_o (3D rotation vector).
    R_des : desired rotation matrix (3x3)
    R     : current rotation matrix (3x3)
    """
    R_err = R_des.T @ R
    # Skew-symmetric part → vee operator
    e_o = 0.5 * pin.log3(R_err)   # log3 returns a 3D rotation vector
    return e_o

# pinocchio compute task space state 3D position and velocity
def compute_task_state(q, dq):
    pin.forwardKinematics(model, data, q, dq)
    pin.updateFramePlacements(model, data)
    
    left_ee_SE3 = data.oMf[left_ee_frame_id].copy()
    right_ee_SE3 = data.oMf[right_ee_frame_id].copy()

    # position 
    left_ee_pos = left_ee_SE3.translation.copy()
    right_ee_pos = right_ee_SE3.translation.copy()

    # Rotation matrix via quaternion (Quaternion.toRotationMatrix = quaternionToMatrix)
    left_ee_quat = np.array(pin.Quaternion(left_ee_SE3.rotation).coeffs())
    right_ee_quat = np.array(pin.Quaternion(right_ee_SE3.rotation).coeffs())

    left_ee_vel = pin.getFrameVelocity(model, data, left_ee_frame_id, pin.ReferenceFrame.LOCAL_WORLD_ALIGNED).copy()
    right_ee_vel = pin.getFrameVelocity(model, data, right_ee_frame_id, pin.ReferenceFrame.LOCAL_WORLD_ALIGNED).copy()

    ee_pos = np.hstack([left_ee_pos, left_ee_quat, right_ee_pos, right_ee_quat ])
    ee_vel = np.hstack([left_ee_vel.linear, left_ee_vel.angular, right_ee_vel.linear, right_ee_vel.angular])

    return ee_pos, ee_vel

# pinocchio compute task space 3D Jacobian and its derivative
def compute_task_Jacobian(q, dq):
    pin.forwardKinematics(model, data, q, dq)
    pin.updateFramePlacements(model, data)

    # detect the end-effector number, and to compute the Jacobian
    # the left end-effector
    pin.computeFrameJacobian(model, data, q, left_ee_frame_id, 
                                                pin.ReferenceFrame.LOCAL_WORLD_ALIGNED)
    left_ee_Jacobian_6d = pin.getFrameJacobian(model, data, left_ee_frame_id, 
                                                pin.ReferenceFrame.LOCAL_WORLD_ALIGNED)

    left_ee_dotJacobian_6d = pin.frameJacobianTimeVariation(model, data, q, dq, left_ee_frame_id,    
                                   pin.ReferenceFrame.LOCAL_WORLD_ALIGNED)
    
    # print("Jacobian_left_ee:\n", Jacobian_left_ee)

    # the right end-effector
    pin.computeFrameJacobian(model, data, q, right_ee_frame_id, 
                                                pin.ReferenceFrame.LOCAL_WORLD_ALIGNED)
    right_ee_Jacobian_6d = pin.getFrameJacobian(model, data, right_ee_frame_id, 
                                                pin.ReferenceFrame.LOCAL_WORLD_ALIGNED)
    
    right_ee_dotJacobian_6d = pin.frameJacobianTimeVariation(model, data, q, dq, right_ee_frame_id, 
                                                             pin.ReferenceFrame.LOCAL_WORLD_ALIGNED)

    Ja_6d = np.vstack([left_ee_Jacobian_6d, right_ee_Jacobian_6d])  # linear velocity part
    dJa_6d = np.vstack([left_ee_dotJacobian_6d, right_ee_dotJacobian_6d])  # linear velocity part

    return Ja_6d, dJa_6d

# Test the Function of compute_task_Jacobian()
# -----------------------------------------#
# Pinocchio forwardKinematics requires numpy arrays (not Python lists)



In [3]:
# Computing Cartesian Space Dynamics in Pinocchio
def compute_Cartesian_space_dynamics(q, dq):
    pin.forwardKinematics(model, data, q, dq)
    pin.updateFramePlacements(model, data)

    # Joint-space mass matrix
    pin.crba(model, data, q)

    M = data.M
    M = 0.5 * (M + M.T)           # symmetrize numerically

    # Nonlinear effects h(q,dq) = C(q,dq) dq + g(q)
    h = pin.nonLinearEffects(model, data, q, dq)

    # Gravity separated if you want it explicitly
    g = pin.computeGeneralizedGravity(model, data, q)

    dns_C = pin.computeCoriolisMatrix(model, data, q, dq)

    # get end-effector Jacobian and its derivative
    Ja, Jadot = compute_task_Jacobian(q, dq)

    # Operational-space inertia Λ = (J M^{-1} Jᵀ)^{-1}
    MinvJt = np.linalg.solve(M, Ja.T)         # (nv×nv)\(nv×12) ⇒ nv×12

    eps = 1e-6
    A = Ja @ MinvJt
    A = 0.5 * (A + A.T)
    Lambda = np.linalg.inv(A + eps * np.eye(A.shape[0]))
    # Lambda = np.linalg.inv(Ja @ MinvJt)       # 12×12

    # Calculate \miu = Lambda( Ja @ M^{-1} @ C - dotJa)\dotq
    JM_inv_C = Ja @ np.linalg.solve(M, dns_C)   # m×nv
    mu = Lambda @ (JM_inv_C - Jadot) # @ dq # shape (m,)

    # Calculate the J_sharp
    J_sharp = np.linalg.solve(M, Ja.T) @ Lambda

    
    return Lambda, mu, J_sharp, g  #Lambda, mu, p

# Test the Function of compute_task_Jacobian()
# -----------------------------------------#
# q = np.random.rand(model.nq)
# dq = np.random.rand(model.nv)
# Lambda, Gamma, F_g = compute_Cartesian_space_dynamics(q, dq)
# -----------------------------------------# 

In [4]:
# Stiffness and Damping Matrices
# end-edffector of both arms 
# K_p = np.diag([100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0]) 
K_p_diag = 30.0
task_size = 12 # 6 DoF for left arm + 6 DoF for right arm
diag_vector = np.full(task_size, K_p_diag)
K_p = np.diag(diag_vector)

# end-edffector of both arms left{x,y,z}, right{x,y,z}
K_d_diag = 60.0
task_size = 12 # 6 DoF for left arm + 6 DoF for right arm
diag_vector = np.full(task_size, K_d_diag)
K_d = np.diag(diag_vector)

# Cartesian Impedance Control Law 
def cartesian_impedance_control(x, dot_x, x_des, dot_x_des, ddot_x_des, Lambda, mu, Ja_sharp):
    # position error
    e_pos_left = x_des[:3] - x[:3]
    e_pos_right = x_des[7:10] - x[7:10]

    # orientation error: quaternions [x, y, z, w] -> rotation matrices
    R_des_left = pin.Quaternion(x_des[3:7]).toRotationMatrix()
    R_left = pin.Quaternion(x[3:7]).toRotationMatrix()
    e_rot_left = rotation_error(R_des_left, R_left)

    R_des_right = pin.Quaternion(x_des[10:14]).toRotationMatrix()
    R_right = pin.Quaternion(x[10:14]).toRotationMatrix()
    e_rot_right = rotation_error(R_des_right, R_right)

    # 12D error: [pos_L(3), rot_L(3), pos_R(3), rot_R(3)]
    e = np.hstack([e_pos_left, e_rot_left, e_pos_right, e_rot_right])

    # 12D velocity error (linear+angular for both arms)
    de = dot_x_des - dot_x

    # Cartesian impedance wrench
    F = Lambda @ ddot_x_des + mu @ Ja_sharp @ dot_x_des + K_d @ de + K_p @ e

    return F  # not torque yet

In [5]:
# --- 5. Compute spatial spring force between end-effectors --- #
# p_rl_0 = p_r - p_l    p_rl_d = (p_r - p_l) 
# e_rot_0 = rotation_error(R_des_left, R_left) 
# F = K_c * (e_d - e_t)

K_c = 10 * np.eye(6)
e_d = [0.0, 1.0, 0.0, 1.03764157, -1.03764174, -2.19134163e-01] # x, y, z, roll, pitch, yaw

def end_effector_spatial_spring_force(x):
    e_p0 = x[0:3] - x[7:10]
    R_left = pin.Quaternion(x[3:7]).toRotationMatrix()
    R_right = pin.Quaternion(x[10:14]).toRotationMatrix()
    e_r0 = rotation_error(R_right, R_left)

    e_0 = np.hstack([e_p0, e_r0])
    e_d[3:6] = e_r0

    F_spring = K_c @ (np.array(e_d) - np.array(e_0))

    F = np.vstack([np.eye(6), -np.eye(6)]) @ F_spring

    if not hasattr(end_effector_spatial_spring_force, "printed"):
        print("the e_0 is:", e_0)
        print("the e_p0 is:", e_p0)
        print("the e_r0 is:", e_r0)
        print("the F is:", F)        
        end_effector_spatial_spring_force.printed = True
    return F

In [ ]:
# --- 6. Compute spatial spring force between end-effectors --- #
# p_rl_0 = p_r - p_l    p_rl_d = (p_r - p_l) 
# e_rot_0 = rotation_error(R_des_left, R_left) 
# F = K_c * (e_d - e_t)

K_c = 10 * np.eye(6)
e_d = [0.0, 1.0, 0.0, 1.03764157, -1.03764174, -2.19134163e-01] # x, y, z, roll, pitch, yaw

def end_effector_spatial_spring_force(x):
    e_p0 = x[0:3] - x[7:10]
    R_left = pin.Quaternion(x[3:7]).toRotationMatrix()
    R_right = pin.Quaternion(x[10:14]).toRotationMatrix()
    e_r0 = rotation_error(R_right, R_left)

    e_0 = np.hstack([e_p0, e_r0])
    e_d[3:6] = e_r0

    F_spring = K_c @ (np.array(e_d) - np.array(e_0))

    F = np.vstack([np.eye(6), -np.eye(6)]) @ F_spring

    if not hasattr(end_effector_spatial_spring_force, "printed"):
        print("the e_0 is:", e_0)
        print("the e_p0 is:", e_p0)
        print("the e_r0 is:", e_r0)
        print("the F is:", F)        
        end_effector_spatial_spring_force.printed = True
    return F

def object_level_impedance_control(x):


In [6]:
# Reference trajectories
from scipy.spatial.transform import Rotation, Slerp

data_mj.qpos[joint_indices] = joint_initial_pos
data_mj.qvel[joint_indices] = np.zeros_like(model_mj.nv)

mujoco.mj_forward(model_mj, data_mj)
mujoco.mj_step(model_mj, data_mj)
viewer.sync()
q0, dq0 = get_pin_state_from_mujoco()

pin.forwardKinematics(model, data, q0, dq0)
pin.updateFramePlacements(model, data)

# get the initial end-effectors' position and velocity
ini_ee_pos, ini_ee_vel = compute_task_state(q0, dq0)


# desired end-effector position
left_offset_ee_pos = [0.1, -0.0, -.45]
left_offset_ee_rpy = [0.0, -0.0, 0.0]  # roll pitch yaw
right_offset_ee_pos = [0.1, 0.0, -0.45] 
right_offset_ee_rpy = [0.0, -0.0, 0.0] 

def set_desired_ee_pos_and_oritation(left_offset_ee_pos, right_offset_ee_pos, left_offset_ee_rpy, right_offset_ee_rpy):
    # initial positions and quats from data (avoids Quaternion type in slices)
    ini_left_ee_pos = data.oMf[left_ee_frame_id].translation.copy()
    ini_right_ee_pos = data.oMf[right_ee_frame_id].translation.copy()
    ini_left_ee_quat = np.array(pin.Quaternion(data.oMf[left_ee_frame_id].rotation).coeffs())
    ini_right_ee_quat = np.array(pin.Quaternion(data.oMf[right_ee_frame_id].rotation).coeffs())

    # get the desired end-effector position and quaternion
    des_left_ee_pos = ini_left_ee_pos + np.array(left_offset_ee_pos)
    des_right_ee_pos = ini_right_ee_pos + np.array(right_offset_ee_pos)
    # RPY -> rotation matrix; desired orientation = initial @ offset
    ini_left_R = data.oMf[left_ee_frame_id].rotation.copy()
    ini_right_R = data.oMf[right_ee_frame_id].rotation.copy()

    #R(r,p,y)=R z (yaw)R y (pitch)R x (roll)
    left_R_offset = pin.rpy.rpyToMatrix(np.array(left_offset_ee_rpy))
    right_R_offset = pin.rpy.rpyToMatrix(np.array(right_offset_ee_rpy))
    des_left_R = ini_left_R @ left_R_offset
    des_right_R = ini_right_R @ right_R_offset
    des_left_ee_quat = np.array(pin.Quaternion(des_left_R).coeffs())
    des_right_ee_quat = np.array(pin.Quaternion(des_right_R).coeffs())
    des_ee_vel = np.zeros_like(ini_ee_vel)
    des_ee_acc = np.zeros_like(ini_ee_vel)

    return des_left_ee_pos, des_right_ee_pos, des_left_ee_quat, des_right_ee_quat, des_ee_vel, des_ee_acc


# In ros the desired_trajectory should get from a topic
def desired_trajectory(t, T_move: float = 4.0):
    # Initialize once
    if t == 0.0 or not hasattr(desired_trajectory, "initialized"):
        desired_trajectory.ini_left_ee_pos = data.oMf[left_ee_frame_id].translation.copy()
        desired_trajectory.ini_right_ee_pos = data.oMf[right_ee_frame_id].translation.copy()

        desired_trajectory.ini_left_ee_quat = np.array(
            pin.Quaternion(data.oMf[left_ee_frame_id].rotation).coeffs()
        )
        desired_trajectory.ini_right_ee_quat = np.array(
            pin.Quaternion(data.oMf[right_ee_frame_id].rotation).coeffs()
        )

        (desired_trajectory.des_left_ee_pos,
         desired_trajectory.des_right_ee_pos,
         desired_trajectory.des_left_ee_quat,
         desired_trajectory.des_right_ee_quat,
         desired_trajectory.des_ee_vel,
         desired_trajectory.des_ee_acc) = set_desired_ee_pos_and_oritation(
            left_offset_ee_pos,
            right_offset_ee_pos,
            left_offset_ee_rpy,
            right_offset_ee_rpy
        )

        desired_trajectory.initialized = True
        # Read stored variables
        
    ini_left_ee_pos = desired_trajectory.ini_left_ee_pos
    ini_right_ee_pos = desired_trajectory.ini_right_ee_pos
    ini_left_ee_quat = desired_trajectory.ini_left_ee_quat
    ini_right_ee_quat = desired_trajectory.ini_right_ee_quat

    des_left_ee_pos = desired_trajectory.des_left_ee_pos
    des_right_ee_pos = desired_trajectory.des_right_ee_pos
    des_left_ee_quat = desired_trajectory.des_left_ee_quat
    des_right_ee_quat = desired_trajectory.des_right_ee_quat

    if t >= T_move:
        x_des = np.hstack([desired_trajectory.des_left_ee_pos, desired_trajectory.des_left_ee_quat, desired_trajectory.des_right_ee_pos, desired_trajectory.des_right_ee_quat])
        return x_des, np.zeros_like(desired_trajectory.des_ee_vel), np.zeros_like(desired_trajectory.des_ee_acc)

    tau = t / T_move
    tau2 = tau * tau
    tau3 = tau2 * tau
    tau4 = tau3 * tau
    tau5 = tau4 * tau

    # position
    left_x_des = ini_left_ee_pos + (des_left_ee_pos - ini_left_ee_pos) * (10*tau3 - 15*tau4 + 6*tau5)
    right_x_des = ini_right_ee_pos + (des_right_ee_pos - ini_right_ee_pos) * (10*tau3 - 15*tau4 + 6*tau5)

    # velocity (derivative of 5th-order poly; angular vel = 0)
    poly_dot = (30*tau2 - 60*tau3 + 30*tau4) / T_move
    dot_left_lin = (des_left_ee_pos - ini_left_ee_pos) * poly_dot
    dot_right_lin = (des_right_ee_pos - ini_right_ee_pos) * poly_dot
    dot_x_des = np.hstack([dot_left_lin, np.zeros(3), dot_right_lin, np.zeros(3)])

    # acceleration
    poly_ddot = (60*tau - 180*tau2 + 120*tau3) / (T_move * T_move)
    ddot_left_lin = (des_left_ee_pos - ini_left_ee_pos) * poly_ddot
    ddot_right_lin = (des_right_ee_pos - ini_right_ee_pos) * poly_ddot
    ddot_x_des = np.hstack([ddot_left_lin, np.zeros(3), ddot_right_lin, np.zeros(3)])

    # quaternion slerp (scipy quat order: x,y,z,w)
    s = 10*tau3 - 15*tau4 + 6*tau5
    slerp_left = Slerp([0.0, 1.0], Rotation.from_quat([ini_left_ee_quat, des_left_ee_quat]))
    slerp_right = Slerp([0.0, 1.0], Rotation.from_quat([ini_right_ee_quat, des_right_ee_quat]))
    left_x_quat_des = slerp_left(s).as_quat()
    right_x_quat_des = slerp_right(s).as_quat()

    x_des = np.hstack([left_x_des, left_x_quat_des, right_x_des, right_x_quat_des])
    return x_des, dot_x_des, ddot_x_des


# control step
def control_step(t):
    # --- 1. read state from Mujoco and map to Pinocchio (Sensor) --- #
    q, dq = get_pin_state_from_mujoco()
        
    x, dot_x = compute_task_state(q, dq)
    
    # --- 2. Kinematics & Jacobians in Pinocchio --- #
    Ja, dotJa = compute_task_Jacobian(q, dq)

    # --- 4. Compute object level control parameters--- #
    Lambda, mu, J_sharp, g= compute_Cartesian_space_dynamics(q, dq)

    # --- 5. Compute spatial spring force between end-effectors --- #
    F_spring = end_effector_spatial_spring_force(x)

    # --- 6. Object level Impedance torque --- #
    x_des, dot_x_des, ddot_x_des = desired_trajectory(t)
    F = cartesian_impedance_control(x, dot_x, x_des, dot_x_des, ddot_x_des, Lambda, mu, J_sharp)

    tau = g + Ja.T @ (F + F_spring)

    data_mj.ctrl[:] = tau

    return x, dot_x

# Test the Function of control_step()
# -----------------------------------------#
# t = 0
# for i, I in enumerate(model.inertias):
#     print(model.names[i], I.mass, I.lever, I.inertia)

# x, dot_x = control_step(t)
# print(x, dot_x)
# -----------------------------------------# 

In [7]:
DT = model_mj.opt.timestep
sim_time = 10.0
steps = int(sim_time / DT)

log_t = []
log_x = []
log_dx = []
log_ddx = []


t = 0.0
mujoco.mj_step(model_mj, data_mj)

for k in range(steps):
    x, dot_x = control_step(t)
    mujoco.mj_step(model_mj, data_mj)
    viewer.sync()
    
    t += DT

    # Log
    log_t.append(k)
    log_x.append(x.copy())
    log_dx.append(dot_x.copy())


left_offset_ee_pos = [0.0, -0.0, 0.0]
left_offset_ee_rpy = [0.0, -0.0, 0.0]  # roll pitch yaw
right_offset_ee_pos = [0.0, 0.0, 0.0] 
right_offset_ee_rpy = [0.0, -0.0, 0.0] 

# t = 0.0
# mujoco.mj_step(model_mj, data_mj)

# for k in range(steps):
#     x, dot_x = control_step(t)
#     mujoco.mj_step(model_mj, data_mj)
#     viewer.sync()
    
#     t += DT

#     # Log
#     log_t.append(k)
#     log_x.append(x.copy())
#     log_dx.append(dot_x.copy())

the e_0 is: [ 7.93892614e-07  1.16702051e+00  1.58734165e-07  9.92202957e-01
  9.92206098e-01 -6.05868842e-02]
the e_p0 is: [7.93892614e-07 1.16702051e+00 1.58734165e-07]
the e_r0 is: [ 0.99220296  0.9922061  -0.06058688]
the F is: [-7.93892614e-06 -1.67020507e+00 -1.58734165e-06  0.00000000e+00
  0.00000000e+00  0.00000000e+00  7.93892614e-06  1.67020507e+00
  1.58734165e-06  0.00000000e+00  0.00000000e+00  0.00000000e+00]
